In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, precision_score, recall_score 
import lightgbm as lgb
import pickle

In [67]:
df = pd.read_csv('../train_csv/train.csv',index_col=0)
rank_map = {1:6,2:5,3:4,4:3,5:2,6:1}
rank_mod_map = {1:6,2:5,3:4,4:0,5:0,6:0}
df_y = df['着'].map(rank_mod_map).astype(int)
df_x = df.drop(['選手名','着'],axis=1)
x_train, x_vali, y_train, y_vali = train_test_split(df_x, df_y, test_size=0.2, shuffle=False)

train_group = x_train['RaceID'].value_counts(sort = False)
val_group = x_vali['RaceID'].value_counts(sort = False)

x_train = x_train.drop('RaceID', axis=1)
x_vali = x_vali.drop('RaceID', axis=1)

In [68]:
df_y[-12:]

712529    0
712530    6
712531    0
712532    4
712533    5
712534    0
712535    6
712536    0
712537    5
712538    0
712539    0
712540    4
Name: 着, dtype: int64

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 712541 entries, 0 to 712540
Data columns (total 19 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   艇番         712541 non-null  int64  
 1   選手登番       712541 non-null  int64  
 2   選手名        712541 non-null  object 
 3   年齢         712541 non-null  int64  
 4   支部         712541 non-null  int64  
 5   体重         712541 non-null  int64  
 6   級別         712541 non-null  int64  
 7   全国勝率       712541 non-null  float64
 8   全国2連率      712541 non-null  float64
 9   当地勝率       712541 non-null  float64
 10  当地2連率      712541 non-null  float64
 11  モーター2連率    712541 non-null  float64
 12  ボート2連率     712541 non-null  float64
 13  RaceID     712541 non-null  object 
 14  場所         712541 non-null  int64  
 15  R          712541 non-null  int64  
 16  着          712541 non-null  int64  
 17  展示         712541 non-null  float64
 18  スタートタイミング  712541 non-null  float64
dtypes: float64(8), int64(9)

In [69]:

lgbm_params =  {
    'task': 'train',
    'boosting_type': 'gbdt',
    'objective': 'lambdarank', #←ここでランキング学習と指定！
    'metric': 'ndcg',   # for lambdarank
    'ndcg_eval_at': [1,2,3],  # 3連単を予測したい
    'max_position': 6,  # 競艇は6位までしかない
    'learning_rate': 0.01, 
    'group_column': 13,
    'min_data': 1,
    'min_data_in_bin': 1,
    'random_state': 777,
    #'num_leaves': 31,
   #'max_depth':35,
}

lgtrain = lgb.Dataset(x_train, y_train,  group=train_group)
lgvalid = lgb.Dataset(x_vali, y_vali,group=val_group)

lgb_clf = lgb.train(
    lgbm_params,
    lgtrain,
    num_boost_round=10000,
    valid_sets=[lgtrain, lgvalid],
    valid_names=['train','valid'],
    early_stopping_rounds=1000,
    verbose_eval=100
)

/Users/tojo/Documents/boat/.boat_env/lib/python3.10/site-packages/lightgbm/engine.py:181: UserWarning: 'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. Pass 'early_stopping()' callback via 'callbacks' argument instead.
  _log_warning("'early_stopping_rounds' argument is deprecated and will be removed in a future release of LightGBM. "
/Users/tojo/Documents/boat/.boat_env/lib/python3.10/site-packages/lightgbm/engine.py:239: UserWarning: 'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose_eval' argument is deprecated and will be removed in a future release of LightGBM. "


[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Unknown parameter: max_position
[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.191318 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2164
[LightGBM] [Info] Number of data points in the train set: 570032, number of used features: 16
[LightGBM] [Warning] Unknown parameter: max_position
Training until validation scores don't improve for 1000 rounds
[100]	train's ndcg@1: 0.702999	train's ndcg@2: 0.728149	train's ndcg@3: 0.763483	valid's ndcg@1: 0.698868	valid's ndcg@2: 0.726278	valid's ndcg@3: 0.761059
[200]	train's ndcg@1: 0.710476	train's ndcg@2: 0.735909	train's ndcg@3: 0.770555	valid's ndcg@1: 0.704601	valid's ndcg@2: 0.731543	valid's ndcg@3: 0.767309
[300]	train's ndcg@1: 0.718072	train's ndcg@2: 0.74186	train's ndcg@3: 0.775277	valid's ndcg@1: 0.711098	valid's ndcg@2: 0.736611	valid's ndcg@3: 0.771605
[400]	train's ndcg

In [70]:
with open('../model/lgb_clf.pickle', 'wb') as f:
    pickle.dump(lgb_clf, f)

In [71]:
y_pred_model = lgb_clf.predict(x_vali ,num_iteration=lgb_clf.best_iteration)

In [72]:
y_pred_model[:12]


array([-0.0655443 , -0.98709027, -0.14698561, -1.619869  , -0.75746355,
       -1.37171881, -3.20136978,  0.72523392, -1.51991001,  0.46590787,
       -0.45074677, -3.09730677])

In [74]:
y_vali[:12]

/var/folders/39/lzc02mxx3d1bksjxyqbhgv9c0000gn/T/ipykernel_23831/767060331.py:1: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  y_vali[:12]


570032    4
570033    4
570034    6
570035    0
570036    5
570037    0
570038    0
570039    4
570040    0
570041    6
570042    5
570043    0
Name: 着, dtype: int64

In [40]:
val_group[:3]

2020-12-27-11-07    1
2020-12-27-11-08    6
2020-12-27-11-09    6
Name: RaceID, dtype: int64

In [164]:
#Validデータ的中率の算出
j = 0
solo_count = 0
doub_count = 0
tri_count = 0
for i in val_group:
    result = y_pred_model[j:j+i] #グループでの順位
    ans = y_vali.iloc[j:j+i].reset_index() #答え
    result1st = np.argmax(result) #予測の1位

    if len(np.where(result==sorted(result)[::-1][0])[0])>1:
        result2nd = np.where(result==sorted(result)[::-1][1])[0][0]
        result3rd = np.where(result==sorted(result)[::-1][1])[0][1]
    else:
        if i > 1:
            result2nd = np.where(result==sorted(result)[::-1][1])[0][0]
        if i > 2:
            result3rd = np.where(result==sorted(result)[::-1][2])[0][0]
    
    #print(int(ans[ans["着"] == ans.max()[1]].index.values))
    ans1st = int(ans[ans["着"]==ans.max()[1]].index.values)
    and_sort = ans.sort_values("着",ascending=False)['着']
    if i > 2:
        a2n = int(and_sort[1])
        a3n = int(and_sort[2])
        
        if len(ans[ans["着"]==a2n].index.values)>1:
            ans2nd = int(ans[ans["着"]==a2n].index.values[0])
            ans3rd = int(ans[ans["着"]==a2n].index.values[1])
        else:
            if i > 1:
                ans2nd = int(ans[ans["着"]==a2n].index.values[0])
            if i > 2:
                ans3rd = int(ans[ans["着"]==a3n].index.values[0])
    
    if ans1st==result1st:
        #print(ans1st,result1st)
        solo_count = solo_count+1

    if i > 1:
        if (ans1st==result1st)&(ans2nd==result2nd):
            doub_count = doub_count+1

    if i > 2:
        if (ans1st==result1st)&(ans2nd==result2nd)&(ans3rd==result3rd):
            tri_count = tri_count+1 

    j = j + i
print("単勝的中率：",round(solo_count/len(val_group)*100,2),"%")
print("２連単的中率：",round(doub_count/len(val_group)*100,2),"%")
print("３連単的中率：",round(tri_count/len(val_group)*100,2),"%")


単勝的中率： 61.41 %
２連単的中率： 25.82 %
３連単的中率： 11.66 %


In [ ]:
TARGET = '220830'
df_test = pd.DataFrame()
df_test = mkdf(TARGET,df_test, test=True)
df_test = std(lencode(df_test))
#df_test = df_test[df_test['場所'] == 13]
test_x = df_test#.drop(['RaceID'],axis=1)

test_group = mkgrp(test_x)
test_x = test_x#.drop(['R'],axis=1)
y_test_pred = lgb_clf.predict(test_x,group=test_group, num_iteration=lgb_clf.best_iteration)
y_test_pred = pd.DataFrame(y_test_pred)
y_pred = test_x.reset_index()
y_pred['pred'] = y_test_pred
import numpy as np
from scipy.stats import rankdata
l = y_pred.shape[0]//6
y_pred['pred_rank'] = 0
for i in range(l):
    p =[y_pred['pred'][i*6+ t] for t in range(6)]
    rp = rankdata(np.array(p))
    for j in range(6):
        y_pred['pred_rank'][i*6 + j] = rp[j]

y_pred.to_csv('../prediction/pred_'+TARGET+'.csv')
places = y_pred['場所'].unique()
place_list = {
            '桐生':'1',
            '戸田':'02',
            '江戸川':'03',
            '平和島':'04',
            '多摩川':'05',
            '浜名湖':'06',
            '蒲郡':'07',
            '常滑':'08',
            '津':'09',
            '三国':'10',
            'びわこ':'11',
            '住之江':'12',
            '尼崎':'13',
            '鳴門':'14',
            '丸亀':'15',
            '児島':'16',
            '宮島':'17',
            '徳山':'18',
            '下関':'19',
            '若松':'20',
            '芦屋':'21',
            '福岡':'22',
            '唐津':'23',
            '大村':'24'}

def get_keys_from_value(d, val):
    return [k for k, v in d.items() if v == val]

from os import makedirs
for p in places:
    if p < 10:
        v = '0' + str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    else:
        v = str(p)
        PLACE = get_keys_from_value(place_list,v)[0]
    makedirs('../prediction/' + TARGET + '/',exist_ok=True)
    with open("../prediction/"+TARGET+"/"+ TARGET+"_"+ PLACE +".txt", "w"):
        pass
    f = open("../prediction/"+TARGET+"/"+ TARGET+"_"+ PLACE +".txt", "w")
    print(PLACE,file=f)
    print(PLACE)
    y_predp = y_pred[y_pred['場所'] == p]
    pl = y_predp.shape[0]//6
    
    print('-----------------------',file=f)
    for i in range(pl):
        
        print('第'+str(i+1)+'R',file=f)
        race_pd = y_predp.iloc[i*6:i*6+6,:]
        first = race_pd[race_pd['pred_rank'] == 1]['艇番'].iloc[0]
        second = race_pd[race_pd['pred_rank'] == 2]['艇番'].iloc[0]
        third = race_pd[race_pd['pred_rank'] == 3]['艇番'].iloc[0]
        print(str(first)+'-'+str(second) +'-' + str(third),file=f)
        print(str(first)+'-'+str(second) +'-' + str(third))
    print('-----------------------\n',file=f)
    print('-----------------------\n')
    f.close()

NameError: name 'pd' is not defined